This script takes land use data by parcel from the municipality of Belo Horizonte's City Hall and performs an initial wrangling.

First, land uses are excessively disaggregated for the intended model. Hence, land use types will be aggregated into umbrella categories, which is also helpful to maintain compatibility throughout the time series.

Next, land uses will be inputted into a H3 hexagonal grid.

Finally, some relevant uses of the area will be superimposed in the data. That was necessary because it fixes some inconsistencies found in the original data for some of the years (in 2011, e.g., the airport was classified as a warehouse, and the landfill was marked as vacant). This is also interesting because it fixates the location of the infrastructures and marks the places where the subnormal agglomerates are.

The final sequence of steps, which is shown below, is the result of a trial and error analysis that had to devise solutions to practical issues specific to the data at hand. During this process, a number of intermediate steps are required, all of which are commented and reflected upon just before implementation.

2017 data source: http://bhmap.pbh.gov.br

Data for 2011 and 2020 have been kindly provided by PRODABEL --- https://prefeitura.pbh.gov.br/prodabel

For the years 2011 and 2020, an investigation in QGIS revealed there to be overlapping land use geometries. However, these arethe exception and most overlaps concerned areas of vacant land, while a minor number of instances were residential and retail/services. Therefore, this is not considered an important issue and the overlay operations handle this adequately.

# Preliminaries

In [ ]:
# Standard library
import os
import pathlib
import re
import unicodedata
import warnings
from dataclasses import dataclass
from typing import Dict, List, Optional, Sequence, Tuple

# Third-party libraries
import contextily as cx
import geobr
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib_scalebar.scalebar import ScaleBar
from pandas.api.types import CategoricalDtype
import plotly.express as px
from tobler.area_weighted import area_interpolate
from tobler.util import h3fy

%matplotlib inline
%config InlineBackend.figure_format='retina'

## Parent Folders

These should of course be adjusted to reflect the appropriate locations in your disk or wherever

In [ ]:
out_folder = os.environ.get('OUT_FOLDER')
out_folder = pathlib.Path(out_folder)
out_folder = out_folder / 'A'

db_folder = os.environ.get('DB_FOLDER')
db_folder = pathlib.Path(db_folder)

## General Purpose

In [ ]:
years = [2011, 2017, 2018, 2020, 2022]

inpaths = [
    (db_folder
     / 'beaga'
     / 'tipologia_uso_ocupacao'
     / f'uso_ocup_{year}.zip')
    for year in years
]

lu = pd.concat(
    gpd.read_file(path).assign(ano=year)
    for path, year in zip(inpaths, years)
)

In [ ]:
# -------------------------- Normalization ------------------------- #

def normalize_text(value: object) -> str:
    """Lowercase, strip accents, collapse spaces; handles nulls."""
    if pd.isna(value):
        return ""
    s = unicodedata.normalize("NFD", str(value).strip().lower())
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", s)


def normalize_series(series: pd.Series) -> pd.Series:
    """Vectorized text normalization for a pandas Series."""
    return series.astype(str).apply(normalize_text)


# ---------------------------- Regex Set --------------------------- #

REGEX = {
    "special_zone": r"\b(?:zeis|aglomerado subnormal|favela|vila)\b",

    "industry": (
        r"\b(?:industria|industrial|fabrica|oficina|"
        r"galp[aã]o\s+industrial)\b"
    ),

    "infrastructure": (
        r"\b(?:aeroporto|aeroclube|aerodromo|hangar|"
        r"estac(?:ao|ao)|terminal)\b"
    ),

    "retail_services": (
        r"\b(?:shopping|galeria|loja|supermercado|hipermercado|"
        r"hote(l|is)|mote(l|is)|apart ?hotel|estacionamento|"
        r"casa de show|comercial|instituicao financeira|"
        r"galp[aã]o(?!\s+industrial))\b"
    ),

    "public_services": (
        r"\b(?:servic(?:os)?\s+public(?:os)?|delegacia|bombeiro|"
        r"hospital|instituicao\s+de\s+ensino|equipamento\s+de\s+saude)\b"
    ),

    "amenities": (
        r"\b(?:parque[s]?|cemiterio|estadio|ginasio|clube(?:s)?|"
        r"instituicao\s+(?:religiosa|cultural)|"
        r"centro\s+de\s+convenc(?:ao|oes))\b"
    ),

    "mixed": r"\b(?:misto|diversificada)\b",

    "residential": (
        r"(?<!nao\s)(?<!nao\s-\s)\b(?:residencial|casa|sobrado|"
        r"edificio\s+residencial|conjunto\s+multifamiliar|familiar)\b"
    ),

    "vacant": r"\blote\s*\+?\s*vago\b",

    "fallback_building": (
        r"\b(?:edificac(?:ao|oes)|edificio|construcao|predio)\b"
    ),

    # For density (existing)
    "res_high": r"\b(?:edificio\s+residencial|conjunto\s+multifamiliar)\b",
    "morf_5a8": r"\b5\s*a\s*8\s*pav",
    "morf_gt8": r"\bmais\s*de\s*8\s*pav",

    # Surgical guards / tokens (broadened + new)
    "non_res_marker": r"\bnao\s+residencial\b",

    # broaden to include "de uso nao residencial"
    "edif_non_res": r"\bedificio\s+(?:nao\s+residencial|"
                    r"de\s+uso\s+nao\s+residencial)\b",

    "cs_res": r"\bcasa/?sobrado\s+residencial\b",
    "cs_non_res": r"\bcasa/?sobrado\s+nao\s+residencial\b",
    "sobrado_token": r"\bsobrado\b",

    # NEW explicit mixed-use building tags
    "edif_res_e_comercio": (
        r"\bedificio\s+residencial\s+e\s+comercio"
        r"(?:\s+(?:e|\/?ou)\s*servicos)?\b"
    ),
    "edif_uso_misto": r"\bedificio\s+de\s+uso\s+misto\b",

    # Retail high-intensity hint (optional cue when morphology missing)
    "retail_high_hint": (
        r"\b(?:shopping|centro\s+comercial|galeria|mall|"
        r"hipermercado|outlet|centro\s+empresarial)\b"
    ),

    # Mixed-use high-intensity hint via explicit building labels
    "mixed_high_hint": (
        r"(?:\bedificio\s+de\s+uso\s+misto\b|"
        r"\bedificio\s+residencial\s+e\s+comercio"
        r"(?:\s+(?:e|\/?ou)\s*servicos)?\b)"
    ),
}

COMPILED_REGEX = {
    key: re.compile(pattern, flags=re.IGNORECASE)
    for key, pattern in REGEX.items()
}


# -------------------------- Mask Engine --------------------------- #

def build_masks(text_series: pd.Series) -> Dict[str, pd.Series]:
    """Build boolean masks from the compiled regex dict."""
    return {
        name: text_series.str.contains(pattern, regex=True, na=False)
        for name, pattern in COMPILED_REGEX.items()
    }


# ----------------------------- Density ---------------------------- #

def infer_density(
    land_use: pd.Series,
    ocp_series: pd.Series,
    morph_series: pd.Series
) -> pd.Series:
    """Infer density for residential/mixed/retail."""
    is_target = land_use.isin({"residential", "mixed", "retail/services"})

    morf_high = COMPILED_REGEX["morf_gt8"]
    morf_mid = COMPILED_REGEX["morf_5a8"]
    res_high = COMPILED_REGEX["res_high"]

    retail_high_hint = COMPILED_REGEX["retail_high_hint"]
    mixed_high_hint = COMPILED_REGEX["mixed_high_hint"]

    # morphology-based high (applies to all target classes)
    cond_morph_high = (
        morph_series.str.contains(morf_high, regex=True, na=False) |
        morph_series.str.contains(morf_mid,  regex=True, na=False)
    )

    # occupation-based residential high
    cond_res_high = ocp_series.str.contains(res_high, regex=True, na=False)

    # retail high-intensity cue when morphology missing
    is_retail = (land_use == "retail/services")
    cond_retail_hint = is_retail & ocp_series.str.contains(
        retail_high_hint, regex=True, na=False
    )

    # mixed-use explicit building labels imply high density
    is_mixed = (land_use == "mixed")
    cond_mixed_hint = is_mixed & ocp_series.str.contains(
        mixed_high_hint, regex=True, na=False
    )

    conditions = [
        is_target & (
            cond_morph_high | cond_res_high | cond_retail_hint | cond_mixed_hint
        ),
        is_target,
    ]
    choices = ["high", "low"]

    density = np.select(conditions, choices, default=pd.NA)
    cat_type = CategoricalDtype(categories=["high", "low"])
    return pd.Series(density, index=land_use.index).astype(cat_type)


# ------------------------- Main Pipeline -------------------------- #

# Final order with explicit CASA/SOBRADO NAO RESIDENCIAL rule
# and new explicit building tags for mixed/retail.
CLASS_ORDER_SAFE = [
    ("industry", "industry"),
    ("infrastructure", "infrastructure"),
    ("public_services", "public services"),
    ("amenities", "amenities"),

    # explicit building-level tags
    ("edif_non_res", "retail/services"),
    ("edif_res_e_comercio", "mixed"),
    ("edif_uso_misto", "mixed"),

    ("retail_services", "retail/services"),
    ("__cs_non_res__", "mixed"),          # explicit CS NAO RESIDENCIAL
    ("__sobrado_mixed__", "mixed"),       # sobrado default → mixed
    ("mixed", "mixed"),
    ("__residential_clean__", "residential"),
    ("special_zone", "residential"),      # keep mapping as per your choice
    ("vacant", "vacant"),
    ("fallback_building", "residential"),
]


def classify_land_use(input_df: pd.DataFrame, **kwargs) -> pd.DataFrame:
    """
    Classify land use with priority USO -> OCUPACAO (for lu_base),
    and MORFOLOGIA only for density. Minimal safeguards added.
    """
    column_map = {
        "uso": "TIPOLOGIA_USO",
        "ocp": "TIPOLOGIA_OCUPACAO",
        "morph": "MORFOLOGIA",
        "out_lu": "lu_base",
        "out_den": "lu_density",
        **kwargs,
    }

    # Normalize text columns
    norm = {
        key: normalize_series(input_df[column_map[key]])
        for key in ["uso", "ocp", "morph"]
    }

    # Masks by column and combined (keeps structure light)
    uso_masks = build_masks(norm["uso"])
    ocp_masks = build_masks(norm["ocp"])
    both_masks = build_masks(norm["uso"] + " " + norm["ocp"])

    # ---- custom guards ------------------------------------------------
    non_res_any = (
        ocp_masks["non_res_marker"] | ocp_masks["edif_non_res"] |
        uso_masks["edif_non_res"]  | both_masks["edif_non_res"]
    )

    # Explicit CASA/SOBRADO NAO RESIDENCIAL → mixed
    cs_non_res_mask = (
        ocp_masks["cs_non_res"] | both_masks["cs_non_res"] |
        (
            (ocp_masks["sobrado_token"] | uso_masks["sobrado_token"]) &
            (ocp_masks["non_res_marker"] | both_masks["non_res_marker"])
        )
    )

    # sobrado defaults to mixed unless explicitly residential
    sobrado_mixed = (
        (ocp_masks["sobrado_token"] | uso_masks["sobrado_token"]) &
        ~ocp_masks["cs_res"] &
        # tolerate spurious 'residential' fired by 'casa/sobrado'
        (~both_masks["residential"] | ocp_masks["cs_non_res"])
    )

    # Residential only when clearly residential and not negated
    residential_clean = (
        (both_masks["residential"] | ocp_masks["cs_res"] |
         uso_masks["residential"]) &
        ~non_res_any &
        ~ocp_masks["cs_non_res"]
    )

    # Register custom masks
    custom_masks = {
        "__cs_non_res__": cs_non_res_mask,
        "__sobrado_mixed__": sobrado_mixed,
        "__residential_clean__": residential_clean,
    }

    masks_all = {**both_masks, **custom_masks}

    # Apply classification rules
    conditions = [masks_all[key] for key, _ in CLASS_ORDER_SAFE]
    choices = [label for _, label in CLASS_ORDER_SAFE]
    land_use = np.select(conditions, choices, default="uncharted")

    # Outputs
    output_df = input_df.copy()
    unique_labels = sorted(list(set(choices)))
    cats = unique_labels + ["uncharted"]

    lu_col = column_map["out_lu"]
    den_col = column_map["out_den"]

    output_df[lu_col] = pd.Series(
        land_use, index=input_df.index
    ).astype(CategoricalDtype(categories=cats))

    # ---- tiny post-hoc safety (belt-and-suspenders) ------------------
    # (i) Residential contradicted by non-res markers → mixed
    mask_bad_res = (
        (output_df[lu_col] == "residential") &
        (
            norm["ocp"].str.contains(COMPILED_REGEX["non_res_marker"],
                                     na=False) |
            norm["ocp"].str.contains(COMPILED_REGEX["edif_non_res"],
                                     na=False) |
            norm["ocp"].str.contains(COMPILED_REGEX["cs_non_res"],
                                     na=False)
        )
    )
    output_df.loc[mask_bad_res, lu_col] = "mixed"

    # (ii) Any remaining explicit CS NAO RESIDENCIAL → mixed
    mask_cs_non_res_any = (
        norm["ocp"].str.contains(COMPILED_REGEX["cs_non_res"], na=False) |
        (
            norm["ocp"].str.contains(COMPILED_REGEX["sobrado_token"],
                                     na=False) &
            norm["ocp"].str.contains(COMPILED_REGEX["non_res_marker"],
                                     na=False)
        )
    )
    output_df.loc[mask_cs_non_res_any, lu_col] = "mixed"

    # Density
    output_df[den_col] = infer_density(
        output_df[lu_col], norm["ocp"], norm["morph"]
    )

    return output_df


In [ ]:
lu = classify_land_use(lu)

In [ ]:
def sanity_checks(df: pd.DataFrame, *, 
                  uso_col="TIPOLOGIA_USO",
                  ocp_col="TIPOLOGIA_OCUPACAO",
                  morph_col="MORFOLOGIA",
                  lu_col="lu_base",
                  den_col="lu_density",
                  sample=8) -> pd.DataFrame:
    """
    Lightweight consistency checks for the land-use classification.

    Returns a dataframe with each check, number of violations, and
    a tiny sample of offending indices. Raises AssertionError only
    for critical failures (missing labels or unknown categories).
    """
    # --- normalize for regex tests (non-destructive) ---
    uso_n  = normalize_series(df[uso_col])
    ocp_n  = normalize_series(df[ocp_col])
    morphn = normalize_series(df[morph_col])

    # --- helpers / masks (reuse your compiled regex) ---
    def m(series, key): 
        return series.str.contains(COMPILED_REGEX[key], na=False)

    is_res_label   = df[lu_col].astype(str).eq("residential")
    is_mixed_label = df[lu_col].astype(str).eq("mixed")
    is_retail_lab  = df[lu_col].astype(str).eq("retail/services")
    is_relevant_den = is_res_label | is_mixed_label | is_retail_lab

    # --- define checks: name -> boolean mask of violations ---
    checks = {
        # 0) Label existence / dtype integrity (critical)
        "0a_missing_lu_labels": df[lu_col].isna(),
        "0b_unknown_lu_labels": ~df[lu_col].astype(str).isin({
            "industry", "infrastructure", "public services", "amenities",
            "retail/services", "mixed", "residential", "vacant",
            "uncharted", "special_zone", "fallback_building"
        }),

        # 1) No 'não residencial' inside rows labelled residential
        "1_non_res_in_residential": is_res_label & (
            m(ocp_n, "non_res_marker") | m(ocp_n, "edif_non_res")
            | m(ocp_n, "cs_non_res")
        ),

        # 2) 'sobrado' without 'residencial' should not be residential
        "2_sobrado_impl_res": is_res_label & (
            (m(ocp_n, "sobrado_token") | m(uso_n, "sobrado_token"))
            & ~m(ocp_n, "cs_res")
        ),

        # 3) Industry/infrastructure should not co-announce residential tokens
        "3_industry_with_res_tokens": df[lu_col].astype(str).eq("industry") & (
            m(uso_n, "residential") | m(ocp_n, "residential")
        ),
        "4_infra_with_res_tokens": df[lu_col].astype(str).eq("infrastructure") & (
            m(uso_n, "residential") | m(ocp_n, "residential")
        ),

        # 4) Vacant lots should not mention building terms
        "5_vacant_with_building": df[lu_col].astype(str).eq("vacant") & (
            m(uso_n, "fallback_building") | m(ocp_n, "fallback_building")
        ),

        # 5) Density defined only for relevant classes
        "6_density_out_of_scope": df[den_col].notna() & ~is_relevant_den,

        # 6) Relevant classes should have a density value
        "7_missing_density_in_scope": is_relevant_den & df[den_col].isna(),
    }

    # --- build compact report ---
    rows = []
    for name, mask in checks.items():
        idx = df.index[mask]
        rows.append({
            "check": name,
            "violations": int(mask.sum()),
            "sample_idx": idx[:sample].tolist(),
        })
    report = pd.DataFrame(rows).sort_values("violations", ascending=False)

    # --- critical assertions (fail fast if completely broken) ---
    if report.loc[report["check"] == "0a_missing_lu_labels", "violations"].iat[0] > 0:
        raise AssertionError("Missing land-use labels detected.")
    if report.loc[report["check"] == "0b_unknown_lu_labels", "violations"].iat[0] > 0:
        raise AssertionError("Unknown land-use labels detected.")

    return report


In [ ]:
# After running classify_land_use(…) into df_out:
report = sanity_checks(lu,
                       uso_col="TIPOLOGIA_USO",
                       ocp_col="TIPOLOGIA_OCUPACAO",
                       morph_col="MORFOLOGIA",
                       lu_col="lu_base",
                       den_col="lu_density")
report


In [ ]:
# Optional: distribution drift by year (quick glance)
pivot = (lu
         .pivot_table(index="ano", columns="lu_base",
                      values="TIPOLOGIA_USO", aggfunc="count", fill_value=0)
         .apply(lambda s: s / s.sum(), axis=1))
display(pivot.round(3))


In [ ]:
def create_geodataframe_from_output(
    classified_df: pd.DataFrame, 
    wkt_column: str = 'GEOMETRIA', 
    crs: str = 'EPSG:31983'
) -> gpd.GeoDataFrame:
    """
    Converts a DataFrame with WKT geometries into a GeoDataFrame.

    This function safely handles malformed or empty WKT strings by 
    identifying and dropping them before attempting conversion.

    Args:
        classified_df: The input DataFrame containing classification
                       results and a WKT geometry column.
        wkt_column: The name of the column containing WKT strings.
                    Defaults to 'GEOMETRIA'.
        crs: The Coordinate Reference System to set for the
             GeoDataFrame. Defaults to 'EPSG:31983' (SIRGAS 2000 / 
             UTM zone 23S).

    Returns:
        A clean GeoDataFrame with a valid geometry column and CRS.
    """
    if wkt_column not in classified_df.columns:
        raise ValueError(
            f"WKT column '{wkt_column}' not found in the DataFrame."
        )

    df = classified_df.copy()

    # --- ROBUSTNESS FIX STARTS HERE ---
    
    # Replace empty strings with actual Null values (NaN)
    df[wkt_column] = df[wkt_column].replace(r'^\s*$', np.nan, regex=True)

    # Identify all rows that have null geometry
    invalid_mask = df[wkt_column].isna()
    invalid_count = invalid_mask.sum()

    if invalid_count > 0:
        print(
            f"⚠️ Found and dropped {invalid_count} row(s) with "
            "null or empty geometries."
        )
        # Keep only the rows with valid geometry strings
        df = df[~invalid_mask]

    # --- FIX ENDS HERE ---

    # Proceed with conversion on the cleaned data
    # 'on_error' is still a good safeguard for other WKT errors
    geometries = gpd.GeoSeries.from_wkt(df[wkt_column], on_invalid='ignore')

    # Create the GeoDataFrame, ensuring index alignment
    gdf = gpd.GeoDataFrame(df, geometry=geometries, crs=crs)
    
    # Final check for any geometries that failed coercion despite cleaning
    final_invalid = gdf.geometry.isna().sum()
    if final_invalid > 0:
        print(
            f"⚠️ Dropped an additional {final_invalid} row(s) with "
            "malformed WKT during final conversion."
        )
        gdf = gdf[gdf.geometry.notna()]

    return gdf

In [ ]:
lu_gdf = create_geodataframe_from_output(lu)

In [ ]:
mask = lu_gdf.TIPOLOGIA_OCUPACAO.str.match('aterro', flags=re.IGNORECASE, na=False)
lu_gdf.loc[mask, 'lu_base'] = 'public services'

# Category Revision

This section concerns some more specific land use (re)classifications that have to be made manually.
- First, I did so that whatever hexagons intersect with the road network will be labeled as infrastructure and are then fixed as a static land use. **This proved not a good approach, as it can convert to infrastructure uses by main roads, which are often where most changes occur and along where several retail lies.**
- Second, some major local landmarks are enforced. That is done especially because of the 2011 data, which missclassified some old and established local landmarks or infrastructures. For example, the local airport was marked as a warehouse, and both some parks and the landfill were marked as vacant. Another example is that of Lake Pampulha, which is understandably not represented by polygons of any kind - somewhere that is not land could not be represented by a land use polygon after all. Nevertheless, I chose to retain that region in the hexgonal grid of the municipality, which are then maked as amenity.
- Third, there are parcels that mostly coincide with the footprints of the subnormal agglomerates in the municipality - see note below. Those are to be labeled subnormal. It remains to be seen whether its worth keeping them separated.

***It should be noted that some vacant plots are preservation areas and cannot be occupied during simulation. Such constrainsts are to be enforced when zoning restrictions are imposed in the model.***


***Note:***

A subnormal agglomerate is a form of irregular occupation of land – either public or private - owned by a third party, for housing purposes in urban areas, usually characterized by an irregular urban pattern, with scarce essential public services and located in areas not proper or allowed for housing use.  In Brazil, those irregular settlements are known by the names of favelas, invaded areas, slums in deep valleys, slums in low-lands, communities, villages, slums in backwaters, irregular lots, shacks and stilt houses

## Reclassifying

***Note***: reassining to infrastructure all cells that touch infrastructure does not work for larger cells sizes. That happens because if greater than a given size, the hexagon will be larger than city blocks and will thus intersect with a street somewhere. Hence, if the problem is to be scalable, this must be handled.

In [ ]:
# ----------------------------- Config -------------------------------- #

@dataclass(frozen=True)
class OsmLandmarkConfig:
    place: str = "Belo Horizonte, MG, Brazil"
    epsg: int = 31983  # your project CRS

    # Priority list: later items overwrite earlier ones on conflicts.
    # (e.g., put static layers after passive layers)
    layers: Tuple[Tuple[str, str, Dict], ...] = (
        ("passive", "vacant", {
            "natural": [
                "wood", "tree_row", "tree", "scrub", "heath", "moor",
                "grassland", "fell", "bare_rock", "scree", "shingle",
                "sand", "mud",
            ],
        }),
        ("static", "amenities", {
            "tourism": "zoo",
            "leisure": ["stadium", "sports_centre", "park", "nature_reserve"],
            "boundary": ["national_park", "protected_area"],
        }),
        ("static", "public services", {
            "name": "UFMG",
        }),
        ("static", "infrastructure", {
            "name": ["Aeroporto", "ETE Onça"],
            "landuse": "landfill",
        }),
    )


# ------------------------- OSM fetch utilities ----------------------- #

def fetch_osm_polygons(place: str, tags: Dict, target_epsg: int) -> gpd.GeoDataFrame:
    """
    Fetch OSM features for a place and return only polygonal geometries in target CRS.
    """
    gdf = ox.features_from_place(place, tags=tags)
    mask_poly = gdf.geom_type.isin(["Polygon", "MultiPolygon"])
    polys = gdf.loc[mask_poly, ["geometry"]].copy()
    return polys.to_crs(target_epsg)


def build_landmark_layers(cfg: OsmLandmarkConfig) -> List[Tuple[str, str, gpd.GeoDataFrame]]:
    """
    Materialize configured landmark layers as polygon GeoDataFrames in order.
    Returns a list of (activity_category, land_use, polygons_gdf).
    """
    out: List[Tuple[str, str, gpd.GeoDataFrame]] = []
    for activity_category, land_use, tags in cfg.layers:
        polys = fetch_osm_polygons(cfg.place, tags, cfg.epsg)
        if not polys.empty:
            out.append((activity_category, land_use, polys))
    return out


# ------------------------- Impose on parcels ------------------------- #

def impose_landmarks_on_parcels(
    parcels: gpd.GeoDataFrame,
    cfg: OsmLandmarkConfig,
    *,
    use_col: str = "lu_base",
    activity_col: Optional[str] = None,       # e.g., "lu_activity" if you keep it
    predicate: str = "intersects",            # polygon-to-polygon
) -> gpd.GeoDataFrame:
    """
    Overwrite parcel classification when a parcel geometry intersects
    OSM landmark polygons. Later layers in the config have higher priority.

    Parameters
    ----------
    parcels : GeoDataFrame
        Parcel polygons with a valid CRS (will be reprojected if needed).
    cfg : OsmLandmarkConfig
        Place, EPSG, and ordered OSM tag layers.
    use_col : str
        Column to overwrite with landmark land-use labels (e.g., 'amenities').
    activity_col : Optional[str]
        If provided, also set activity category (e.g., 'static', 'passive').
    predicate : str
        Spatial join predicate; 'intersects' is standard for polygons.
    """
    # Ensure CRS
    gdf = parcels.to_crs(cfg.epsg) if (parcels.crs is None or parcels.crs.to_epsg() != cfg.epsg) else parcels.copy()

    # Initialize columns if missing
    if use_col not in gdf.columns:
        gdf[use_col] = None
    if activity_col and activity_col not in gdf.columns:
        gdf[activity_col] = None

    # Iterate in priority order: last layer wins
    for activity_category, land_use, polys in build_landmark_layers(cfg):
        if polys.empty:
            continue

        joined = gpd.sjoin(
            gdf[["geometry"]], polys[["geometry"]],
            how="inner", predicate=predicate
        )
        if joined.empty:
            continue

        affected = joined.index.unique()
        gdf.loc[affected, use_col] = land_use
        if activity_col:
            gdf.loc[affected, activity_col] = activity_category

    return gdf


# --------------------------- One-shot helper ------------------------- #

def reclassify_parcels_with_osm(
    parcels: gpd.GeoDataFrame,
    *,
    place: str = "Belo Horizonte, MG, Brazil",
    epsg: int = 31983,
    use_col: str = "lu_base",
    activity_col: Optional[str] = None,
) -> gpd.GeoDataFrame:
    """
    Convenience wrapper that uses the default layer set and order.
    """
    cfg = OsmLandmarkConfig(place=place, epsg=epsg)
    return impose_landmarks_on_parcels(
        parcels, cfg, use_col=use_col, activity_col=activity_col
    )


In [ ]:
lu_gdf = reclassify_parcels_with_osm(
    lu_gdf,
    place="Belo Horizonte, MG, Brazil",
    epsg=31983,
    use_col="lu_base",          # overwrite parcel land-use when OSM hits
    #activity_col="lu_activity", # optional, if you track static/active/passive
)

In [ ]:
lu_gdf.loc[(lu_gdf.ID_TP_USO_OCP == '328391') & (lu_gdf.ano == 2011), 'lu_base'] = 'vacant'

In [ ]:
lu_gdf.to_parquet(
    out_folder / 'land_uses/land_use_by_parcel_and_year.parquet'
    )

# Parcel Maps

In [ ]:
def plot_land_use_summary(
    gdf: gpd.GeoDataFrame,
    land_use_col: str = "lu_base",
    year_col: str = "ano",
    palette: dict | None = None,
    *,
    category_order: list[str] | None = None,   # custom order for land-use categories
    a4_ppi: int = 96,                          # pixels per inch for A4 width
    height_px: int = 600,                      # adjustable height in pixels
    legend_reverse: bool = False,
    title: str = "Land Use Composition by Year",
):
    """
    Stacked area composition by year with A4 width, adjustable height, no segment lines,
    and custom category order.
    """
    df = gdf.copy()
    df["area_ha"] = df.geometry.area / 10_000

    # aggregate
    summary = (
        df.groupby([year_col, land_use_col], as_index=False)["area_ha"]
          .sum()
    )
    summary[year_col] = summary[year_col].astype(str)

    # figure size: A4 width in pixels
    width_px = int(round(8.27 * a4_ppi))

    # category ordering for stack & legend
    cat_orders = {land_use_col: category_order} if category_order else None

    fig = px.bar(
        summary,
        x=year_col,
        y="area_ha",
        color=land_use_col,
        color_discrete_map=palette,
        category_orders=cat_orders,
        barmode="relative",
        title=title,
        width=width_px,
        height=height_px,
    )

    # styling
    fig.update_traces(marker_line_width=0)  # no lines separating categories
    fig.update_layout(
        xaxis_title="Ano",
        yaxis_title="Área Total (ha)",
        legend_title="Uso do Solo",
        bargap=0.15,
        template="simple_white",
        title=dict(x=0.5, xanchor="center"),
        legend_traceorder="reversed" if legend_reverse else "normal",
    )
    fig.update_yaxes(tickformat=",")

    return fig

In [ ]:
def save_plotly_figure(
    fig,
    path: str | pathlib.Path,
    *,
    scale: int = 2,                  # raster upscaling for static images
    include_plotlyjs: str = "cdn",   # for HTML export
) -> pathlib.Path:
    """
    Save a Plotly figure to the given path.
    - .html  -> interactive HTML
    - .png/.jpg/.jpeg/.webp/.svg/.pdf -> static image (requires 'kaleido')
    Returns the resolved Path to the saved file.
    """
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)

    ext = p.suffix.lower()
    if ext == ".html":
        fig.write_html(str(p), include_plotlyjs=include_plotlyjs, full_html=True)
    elif ext in {".png", ".jpg", ".jpeg", ".webp", ".svg", ".pdf"}:
        fig.write_image(str(p), scale=scale)
    else:
        # default to PNG if no/unknown extension
        p = p.with_suffix(".png")
        fig.write_image(str(p), scale=scale)
    return p

In [ ]:
renaissance_palette = {
    'residential': '#F1E8B8',      # deeper brown, like “burnt umber”
    'retail/services': '#8B2D2D',  # vermelho tijolo (atraente, destaca)
    'mixed': '#9D571B',            # a lighter, more golden terracotta
    'industry': '#60435D',         # verde-acinzentado, pesado/industrial
    'public services': '#426C8A',  # azul-violeta, institucional
    'infrastructure': '#224259',   # azul aço, técnico
    'amenities': '#16271B',        # verde musgo, remete a praças/jardins
    'vacant': '#A6A659',           # verde-oliva claro, aspecto árido
    'uncharted': '#CFCFCF',        # cinza neutro, discretíssimo
}

In [ ]:
fig = plot_land_use_summary(
    lu_gdf,
    palette=renaissance_palette, 
    category_order=list(renaissance_palette.keys())
    )

save_plotly_figure(fig, out_folder / f'figs/land_use_by_hectares.png') 

In [ ]:
def plot_land_use_map(
    gdf: gpd.GeoDataFrame,
    target_year: int,
    palette: dict,
    land_use_col: str = 'lu_base',
    year_col: str = 'ano',
    output_filename: str = None
):
    """
    Generates a publication-quality land use map for a specific year,
    formatted to A4 width with a guaranteed custom legend.

    Args:
        gdf: GeoDataFrame with a projected CRS (e.g., UTM).
        target_year: The year to plot.
        palette: Dictionary mapping land uses to colors.
        land_use_col: Column with land use categories.
        year_col: Column with year data.
        output_filename: Filename to save the map.
    """
    # Filter the GeoDataFrame for the target year
    year_gdf = gdf[gdf[year_col] == target_year].copy()
    if year_gdf.empty:
        print(f"⚠️ No data for year {target_year}. Cannot create map.")
        return

    # Calculate A4-proportional figure size
    minx, miny, maxx, maxy = year_gdf.total_bounds
    aspect_ratio = (maxy - miny) / (maxx - minx)
    a4_width_inches = 8.27
    figure_height = a4_width_inches * aspect_ratio

    # Create the plot
    fig, ax = plt.subplots(
        1, 1, figsize=(a4_width_inches, figure_height)
    )

    # Plot the land use polygons using the explicit color list
    year_gdf.plot(
        ax=ax,
        linewidth=0,
        color=[palette.get(cat, '#FFFFFF') for cat in year_gdf[land_use_col]]
    )
    
    # --- MANUAL LEGEND CREATION ---
    # Create custom legend handles (color patches) from the palette
    legend_handles = [
        Patch(facecolor=color, edgecolor='k', label=label)
        for label, color in palette.items()
        if label in year_gdf[land_use_col].unique() # Only show used categories
    ]
    ax.legend(
        handles=legend_handles,
        title="Land Use",
        bbox_to_anchor=(1.05, 1),
        loc='upper left'
    )

    # Add basemap and cartographic elements
    cx.add_basemap(
        ax, crs=year_gdf.crs.to_string(), source=cx.providers.CartoDB.Positron
    )
    ax.add_artist(ScaleBar(1, "m", location="lower left"))

    # Add North arrow
    x, y, arrow_length = 0.95, 0.95, 0.1
    ax.annotate(
        'N', xy=(x, y), xytext=(x, y - arrow_length),
        arrowprops=dict(facecolor='black', width=5, headwidth=15),
        ha='center', va='center', fontsize=20, xycoords=ax.transAxes
    )

    # Final styling and save
    ax.set_title(f'Land Use Map for {target_year}', fontsize=16, pad=20)
    ax.set_axis_off()

    if output_filename is not None:
        plt.savefig(output_filename, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"✅ Map for year {target_year} saved to '{output_filename}'")
    else:
        plt.show()

In [ ]:
for yr in [2011, 2017, 2018, 2020, 2022]:
    plot_land_use_map(
        lu_gdf,
        target_year=yr,
        palette=renaissance_palette,
        output_filename=out_folder / f'maps/land_use_by_parcel-{yr}.png'
    )

## Retrieving H3 Hexagons

The resolution of the hexagons is defined in a way that its area is close to the median area of a land parcel in Belo Horizonte.

In [ ]:
def get_hexagon_edge_length(land_uses):
    """Computes the edge of an hexagon with the same area as that
    of the median parcel. It considers only active uses because
    both passive and static include may include quite large areas.
    
    Requires projected CRS.
    """
    view = land_uses
    mean_area = np.mean(view.area)
    edge_length = np.sqrt(mean_area * 2 / (3 * np.sqrt(3)))

    print(f'Hex edge should be of approximately {edge_length:.2f} meters')

get_hexagon_edge_length(lu_gdf)

That leaves either H3 resolution 11 (edge length of approx. 25m) or resolution 12 (~9.5m) — _refer to https://h3geo.org/docs/core-library/restable/ for H3 resolutions_

I'll adopt ***Resolution 11*** for the time being, as memory is a concern. But I will also make the land use imputations in hexes 9 and 10, just in caseI run into memory problems in the future.

# Imputações

In [ ]:
# =============================== config =============================== #

@dataclass(frozen=True)
class H3ImputeConfig:
    """
    Minimal configuration for land-use → H3 imputation in a single-user
    notebook context (no extra guardrails).
    """
    h3_resolution: int
    equal_area_epsg: int = 31983
    land_use_col: str = "lu_base"
    density_col: str = "lu_density"
    year_col: Optional[str] = 'ano'
    density_relevant: frozenset[str] = frozenset(
        {"residential", "mixed", "retail/services"}
    )
    # e.g., {"residential": {"low": 1.0, "high": 1.8}, ...}
    weights: Dict[str, Dict[str, float]] = None
    # Optional chunking for very large grids
    chunk_size: Optional[int] = None
    # If provided, must already be indexed by 'hex_id'
    prebuilt_hexes: Optional[gpd.GeoDataFrame] = None


# ================================ helpers ============================ #

def _to_equal_area(gdf: gpd.GeoDataFrame, epsg: int) -> gpd.GeoDataFrame:
    if gdf.crs and gdf.crs.to_epsg() == epsg:
        return gdf
    return gdf.to_crs(epsg)


def _build_h3_grid(cfg: H3ImputeConfig) -> gpd.GeoDataFrame:
    """
    Build one H3 grid from the convex hull of the study area.
    Assumes h3fy outputs a 'hex_id' column.
    """
    if cfg.prebuilt_hexes is not None:
        return cfg.prebuilt_hexes

    study_geom = geobr.read_municipality(
        code_muni=3106200, simplified=False
        ).to_crs(cfg.equal_area_epsg)
    
    return h3fy(study_geom, resolution=cfg.h3_resolution)


def _prepare_extensive_variables(
    parcels_eq: gpd.GeoDataFrame,
    class_labels: Sequence[str],
    cfg: H3ImputeConfig,
) -> Tuple[pd.DataFrame, List[str], List[str]]:
    """
    Create extensive variables for interpolation:
      - area::<class> : polygon area (hectares)
      - eff::<class>  : density-weighted area (hectares)
    """
    df = parcels_eq[[cfg.land_use_col, cfg.density_col, "geometry"]].copy()
    df["area_m2"] = (df.geometry.area).astype("float32")

    land_use = df[cfg.land_use_col].astype(str)
    density = df[cfg.density_col].astype(str)

    area_columns: List[str] = []
    effective_area_columns: List[str] = []
    weights_cfg = cfg.weights or {}

    for lu_class in class_labels:
        w_low = float(weights_cfg.get(lu_class, {}).get("low", 1.0))
        w_high = float(weights_cfg.get(lu_class, {}).get("high", 1.0))

        area_col = f"area::{lu_class}"
        eff_col = f"eff::{lu_class}"
        is_target_class = (land_use == lu_class)

        df[area_col] = np.where(is_target_class, df["area_m2"], 0.0).astype(
            "float32"
        )

        if lu_class in cfg.density_relevant:
            effective_area = np.where(
                is_target_class & (density == "high"),
                w_high * df["area_m2"],
                np.where(
                    is_target_class & (density == "low"),
                    w_low * df["area_m2"],
                    0.0,
                ),
            )
        else:
            effective_area = df[area_col]

        df[eff_col] = effective_area.astype("float32")

        area_columns.append(area_col)
        effective_area_columns.append(eff_col)

    keep = ["geometry"] + area_columns + effective_area_columns
    return df[keep], area_columns, effective_area_columns


def _argmax_and_margin(matrix: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Return argmax per row and (top - second) margin per row.
    """
    if matrix.size == 0:
        return np.array([]), np.array([])
    order = np.argsort(matrix, axis=1)
    top_idx = order[:, -1]
    sec_idx = order[:, -2] if matrix.shape[1] > 1 else np.full(len(top_idx), -1)
    top_val = matrix[np.arange(len(top_idx)), top_idx]
    sec_val = np.where(sec_idx >= 0,
                       matrix[np.arange(len(top_idx)), sec_idx], 0.0)
    return top_idx, (top_val - sec_val)


def _row_entropy(probabilities: np.ndarray) -> np.ndarray:
    """
    Shannon entropy per row; NaN where the row sum is zero.
    """
    with np.errstate(divide="ignore", invalid="ignore"):
        safe = np.where(probabilities > 0, probabilities, 1.0)
        entropy = -(probabilities * np.log(safe)).sum(axis=1)
    empty = np.isclose(probabilities.sum(axis=1), 0.0)
    entropy[empty] = np.nan
    return entropy


def _qc_area_totals(
    src_df: pd.DataFrame,
    hexes: gpd.GeoDataFrame,
    area_columns: list[str],
    class_labels: Sequence[str],
) -> pd.DataFrame:
    """
    Compare source vs. interpolated total area (hectares) by class.
    """
    src_totals = src_df[area_columns].sum(axis=0).to_numpy(dtype=float)
    est_totals = hexes[area_columns].sum(axis=0).to_numpy(dtype=float)
    out = pd.DataFrame(
        {
            "class": list(class_labels),
            "area_src_ha": src_totals,
            "area_interp_ha": est_totals,
            "diff_ha": est_totals - src_totals,
        }
    )
    return out.sort_values("class", ignore_index=True)


# ======================== core on a target grid ======================= #

def _impute_on_grid(
    hex_grid: gpd.GeoDataFrame,
    src_df: pd.DataFrame,
    area_columns: list[str],
    effective_area_columns: list[str],
    class_labels: Sequence[str],
) -> gpd.GeoDataFrame:
    """
    Interpolate extensive variables; compute raw/weighted modal classes,
    margins, and entropy.
    """
    hexes = area_interpolate(
        source_df=src_df,
        target_df=hex_grid,
        extensive_variables=area_columns + effective_area_columns,
        intensive_variables=None,
        allocate_total=False,
        n_jobs=-1,
    )

    area_matrix = hexes[area_columns].to_numpy(dtype="float32")
    effective_area_matrix = hexes[effective_area_columns].to_numpy(
        dtype="float32"
    )

    total_area_m2 = area_matrix.sum(axis=1, dtype="float64")
    shares_by_area = np.divide(
        area_matrix, total_area_m2[:, None], where=total_area_m2[:, None] > 0.0
    )

    top_index_area, margin_area = _argmax_and_margin(shares_by_area)
    class_labels_arr = np.array(class_labels, dtype=object)

    hexes["lu_mode_raw"] = np.where(
        total_area_m2 > 0, class_labels_arr[top_index_area], "no_data"
    )
    hexes["lu_margin_raw"] = np.where(total_area_m2 > 0, margin_area, np.nan)
    hexes["lu_entropy"] = _row_entropy(shares_by_area)

    total_effective_area = effective_area_matrix.sum(axis=1, dtype="float64")
    shares_by_effective_area = np.divide(
        effective_area_matrix,
        total_effective_area[:, None],
        where=total_effective_area[:, None] > 0.0,
    )

    top_index_effective, margin_effective = _argmax_and_margin(
        shares_by_effective_area
    )

    hexes["lu_mode_weighted"] = np.where(
        total_effective_area > 0,
        class_labels_arr[top_index_effective],
        "no_data",
    )
    hexes["lu_margin_weighted"] = np.where(
        total_effective_area > 0, margin_effective, np.nan
    )

    return hexes


# =========================== single fixed grid ======================= #

def _impute_single_grid(
    parcels_eq: gpd.GeoDataFrame,
    hex_grid: gpd.GeoDataFrame,
    cfg: H3ImputeConfig,
) -> Tuple[gpd.GeoDataFrame, pd.DataFrame]:
    class_labels = sorted(parcels_eq[cfg.land_use_col].astype(str).unique())
    lu_dtype = CategoricalDtype(categories=class_labels, ordered=True)

    parcels_eq = parcels_eq.copy()
    parcels_eq[cfg.land_use_col] = parcels_eq[cfg.land_use_col].astype(lu_dtype)

    src_df, area_cols, eff_cols = _prepare_extensive_variables(
        parcels_eq, class_labels, cfg
    )

    if cfg.chunk_size and len(hex_grid) > cfg.chunk_size:
        parts: List[gpd.GeoDataFrame] = []
        for i in range(0, len(hex_grid), cfg.chunk_size):
            target_slice = hex_grid.iloc[i : i + cfg.chunk_size]
            part = _impute_on_grid(
                target_slice, src_df, area_cols, eff_cols, class_labels
            )
            parts.append(part)
        hexes = gpd.GeoDataFrame(
            pd.concat(parts, axis=0), geometry="geometry", crs=hex_grid.crs
        )
    else:
        hexes = _impute_on_grid(
            hex_grid, src_df, area_cols, eff_cols, class_labels
        )

    qc_table = _qc_area_totals(src_df, hexes, area_cols, class_labels)

    hexes["h3_resolution"] = cfg.h3_resolution
    hexes["area_epsg"] = cfg.equal_area_epsg
    return hexes, qc_table


# =========================== public orchestrator ===================== #

def impute_land_use_to_h3(
    parcels: gpd.GeoDataFrame, cfg: H3ImputeConfig
) -> Tuple[gpd.GeoDataFrame, Optional[pd.DataFrame]]:
    """
    Orchestrator:
      1) Reproject to equal-area CRS.
      2) Build H3 grid from convex hull (or reuse prebuilt).
      3) Interpolate once, or once per year if `year_col` is set.
    Returns (hexes, qc_table). Hexes are indexed by 'hex_id'.
    """
    parcels_eq = _to_equal_area(parcels, cfg.equal_area_epsg).copy()
    hex_grid = cfg.prebuilt_hexes or _build_h3_grid(cfg)

    if cfg.year_col is None:
        return _impute_single_grid(parcels_eq, hex_grid, cfg)

    hex_frames: List[gpd.GeoDataFrame] = []
    qc_frames: List[pd.DataFrame] = []

    for year in sorted(parcels_eq[cfg.year_col].dropna().unique()):
        subset = parcels_eq[parcels_eq[cfg.year_col] == year]
        if subset.empty:
            continue
        hx, qc = _impute_single_grid(subset, hex_grid, cfg)
        hx = hx.copy()
        hx[cfg.year_col] = year
        hex_frames.append(hx)

        qc2 = qc.copy()
        qc2["year"] = year
        qc_frames.append(qc2)

    hexes = (
        gpd.GeoDataFrame(
            pd.concat(hex_frames, axis=0), geometry="geometry", crs=hex_grid.crs
        )
        if hex_frames
        else gpd.GeoDataFrame()
    )
    qc_table = pd.concat(qc_frames, ignore_index=True) if qc_frames else None
    return hexes, qc_table


In [ ]:
cfg = H3ImputeConfig(
    h3_resolution=9,
    weights={
        "residential": {"low": 1.0, "high": 2},
        'retail/services': {"low": 1.0, "high": 2},
        'mixed': {"low": 1.0, "high": 2},
        },
    chunk_size=50_000,              # chunk target hexes if large
)

parcels_eq = lu_gdf.to_crs(cfg.equal_area_epsg)  # quick, notebook style
hexes, qc = impute_land_use_to_h3(parcels_eq, cfg)


In [ ]:
hexes.sample()

In [ ]:
qc

In [ ]:
# Defaults you can tune
DEFAULT_DOMINANT_USES = (
    "infrastructure",
    "public services",
    "industry",
    "amenities",
)

def enforce_dominant_use_across_years(
    df: pd.DataFrame,
    *,
    hex_id_column: str = "hex_id",
    year_column: str = "ano",
    land_use_column: str = "lu_mode_weighted",
    dominant_uses: tuple[str, ...] = DEFAULT_DOMINANT_USES,
    priority: tuple[str, ...] | None = None,
    out_column: str = "lu_with_dominance",
) -> pd.DataFrame:
    """
    If any dominant land-use appears in a hex in any year, assign that
    (single) dominant class to *all* rows of the hex. Resolve multiple
    dominant classes by a fixed priority order. Otherwise keep original.

    Returns a copy with one new column (out_column).
    """
    if year_column not in df.columns:
        raise ValueError(
            f"'{year_column}' must be present; this function enforces a "
            "panel-wide rule across years."
        )
    if hex_id_column not in df.columns:
        raise ValueError(f"'{hex_id_column}' must be present.")
    if land_use_column not in df.columns:
        raise ValueError(f"'{land_use_column}' must be present.")

    priority = priority or dominant_uses
    dominant_set = set(dominant_uses)

    # 1) For each hex, collect which dominant uses ever appeared
    is_dominant = df[land_use_column].astype(str).isin(dominant_set)
    pairs = df.loc[is_dominant, [hex_id_column, land_use_column]]

    def choose_override(series: pd.Series):
        """Pick the highest-priority class among those observed in the hex."""
        observed = set(series.astype(str).unique())
        for cls in priority:
            if cls in observed:
                return cls
        return pd.NA

    override_by_hex = (
        pairs.groupby(hex_id_column, sort=False)[land_use_column]
        .apply(choose_override)
    )

    # 2) Broadcast override and finalize output
    out = df.copy()
    override = out[hex_id_column].map(override_by_hex)
    out[out_column] = np.where(
        override.notna(), override, out[land_use_column].astype(str)
    ).astype(pd.Categorical)

    return out


In [ ]:
hexes = enforce_dominant_use_across_years(
    hexes.reset_index(),
    hex_id_column="hex_id",
    year_column="ano",
    land_use_column="lu_mode_weighted",
    out_column="lu_mode_with_dominance",
)

In [ ]:
def add_density_shares_for_all_classes(
    hexes: gpd.GeoDataFrame,
    cfg: H3ImputeConfig,
    *,
    threshold: float = 0.5,
    high_prefix: str = "share_high::",
    low_prefix: str = "share_low::",
    tag_prefix: str = "density::",
) -> gpd.GeoDataFrame:
    """
    For each class in cfg.density_relevant, compute:
      - share_high::<class>  ∈ [0,1] (NaN if indeterminate)
      - share_low::<class>   = 1 - share_high::<class>
      - density::<class>     ∈ {'high','low'} (NaN if indeterminate)

    Uses hex-level interpolated columns:
      area::<class> = A_c
      eff::<class>  = E_c = w_H*H_c + w_L*L_c, with A_c = H_c + L_c
    => H_c = (E_c - w_L*A_c)/(w_H - w_L), clipped to [0, A_c].
    """
    gdf = hexes.copy()

    # classes present as columns and marked density-relevant
    density_classes = [
        cls for cls in cfg.density_relevant
        if (f"area::{cls}" in gdf.columns and f"eff::{cls}" in gdf.columns)
    ]
    if not density_classes:
        return gdf  # nothing to add

    # matrices of areas (A) and effective areas (E)
    area_matrix = gdf[[f"area::{c}" for c in density_classes]].to_numpy("float64")
    effective_area_matrix = gdf[[f"eff::{c}" for c in density_classes]].to_numpy("float64")

    # per-class weights
    weight_low = np.array(
        [float((cfg.weights or {}).get(c, {}).get("low", 1.0)) for c in density_classes],
        dtype="float64",
    )
    weight_high = np.array(
        [float((cfg.weights or {}).get(c, {}).get("high", 1.0)) for c in density_classes],
        dtype="float64",
    )
    weight_delta = weight_high - weight_low

    # solve H and derive shares
    with np.errstate(divide="ignore", invalid="ignore"):
        high_component_area = np.where(
            np.abs(weight_delta) > 1e-12,
            (effective_area_matrix - weight_low * area_matrix) / weight_delta,
            np.nan,
        )
        # numeric safety
        high_component_area = np.clip(high_component_area, 0.0, area_matrix)
        share_high_matrix = np.divide(
            high_component_area, area_matrix,
            out=np.full_like(high_component_area, np.nan),
            where=area_matrix > 0.0
        )
        share_low_matrix = 1.0 - share_high_matrix

    # write columns per class
    for j, cls in enumerate(density_classes):
        gdf[f"{high_prefix}{cls}"] = share_high_matrix[:, j]
        gdf[f"{low_prefix}{cls}"] = share_low_matrix[:, j]

        # tag: 'high' if share_high >= threshold, 'low' if < threshold, NaN if share is NaN
        tag = pd.Series(pd.NA, index=gdf.index, dtype="object")
        sh = share_high_matrix[:, j]
        tag.loc[np.isfinite(sh) & (sh >= threshold)] = "high"
        tag.loc[np.isfinite(sh) & (sh <  threshold)] = "low"
        gdf[f"{tag_prefix}{cls}"] = tag.astype(CategoricalDtype(categories=["high", "low"]))

    return gdf


In [ ]:
hexes = add_density_shares_for_all_classes(hexes, cfg, threshold=0.5)
# e.g., map only residential density:
#   mask = hexes["lu_mode_weighted"] == "residential"
#   color by hexes.loc[mask, "density::residential"]

In [ ]:
hexes.T

# Cleaning Up

In [ ]:
hexes = hexes.reindex(
    columns=[
        'hex_id',
        'h3_resolution',
        'ano',
        'lu_mode_with_dominance',
        'lu_margin_weighted',
        'lu_entropy',
        *hexes.filter(like='density').columns,
        *hexes.filter(like='share').columns,
        'geometry',
    ]
)

In [ ]:
def collapse_density_to_dominant(
    df: pd.DataFrame,
    *,
    dominant_use_col: str = "lu_mode_with_dominance",
    density_prefix: str = "density::",
    out_col: str = "density",
    drop_source_cols: bool = True,
) -> pd.DataFrame:
    """
    Create a single 'density' column by picking, per row, the density value
    from the column 'density::<dominant_use>'. If the dominant use does not
    have a density column, result is NA. Optionally drops the source density
    and share columns.
    """
    out = df.copy()

    # 1) Subset and normalize the density columns to class names
    density_cols = [c for c in out.columns if c.startswith(density_prefix)]
    density_wide = out[density_cols].rename(
        columns=lambda c: c.split("::", 1)[1]
    )

    # 2) Row-wise take: for each row, pick the column matching dominant use
    classes = density_wide.columns
    class_indexer = classes.get_indexer(out[dominant_use_col].astype(str))
    arr = density_wide.to_numpy()

    row_idx = np.arange(len(out))
    picked = np.where(
        class_indexer >= 0, arr[row_idx, class_indexer], pd.NA
    )

    out[out_col] = pd.Series(picked, index=out.index, dtype="string")

    # 3) Optionally drop detailed density/share columns
    if drop_source_cols:
        drop_cols = out.filter(regex=r"^(density::|share_(high|low)::)").columns
        out = out.drop(columns=drop_cols)

    return out

In [ ]:
hexes = collapse_density_to_dominant(
    hexes,  # your table
    dominant_use_col="lu_mode_with_dominance",
    out_col="density",            # will be "low"/"high"/<NA>
    drop_source_cols=True,        # removes density::* and share_*::*
)

hexes.sample(15)

In [ ]:
land_use_categories = {
    'infrastructure': 'static',
    'retail/services': 'active',
    'residential': 'active',
    'amenities': 'active',
    'uncharted': 'passive',
    'vacant': 'passive',
    'public services': 'static',
    'no_data': 'passive',
    'industry': 'static',
    'mixed': 'active'
}
hexes = hexes.loc[hexes.lu_mode_with_dominance != 'no_data']
hexes['categoria'] = hexes.lu_mode_with_dominance.map(land_use_categories)
hexes = hexes.rename(columns={
    'lu_mode_with_dominance': 'uso_do_solo',
    'categoria': 'categoria_uso_do_solo',
    'density': 'densidade',
    'lu_entropy': 'entropia_uso_do_solo',
    'lu_margin_weighted': 'margem_dominancia',
    'h3_resolution': 'aperture',
})

In [ ]:
for yr in [2011, 2017, 2018, 2020, 2022]:
    plot_land_use_map(
        hexes,
        target_year=yr,
        palette=renaissance_palette,
        land_use_col='uso_do_solo',
        output_filename=out_folder / f'maps/land_use_by_hexagon-{yr}.png'
    )

In [ ]:
lu.loc[lu.lu_base == 'retail/services'].TIPOLOGIA_OCUPACAO.value_counts()

# Backup

In [ ]:
outpath = out_folder / 'land_uses/uses_by_hex_r9.parquet'

hexes.to_parquet(outpath)